# Export TALSIM KPRO Block → CSV

**Purpose:** Extracts the `[KPRO]` simulation programme table from a TALSIM
`.PRO` project file and saves it as a clean, semicolon-delimited CSV — useful
for inspecting and documenting all scenario definitions.

**What it does:**
- Reads the `.PRO` file (CP-1252 encoding)
- Locates the `[KPRO]` block and skips frame lines and comment rows
- Writes 13 named columns to CSV:
  `ID`, `OBID`, `dtA`, `dtE`, `dt_min`, `SimStart`, `t_hhmm`,
  `D_mm`, `NID_min`, `HWID`, `tSkal`, `QSkal`, `Beschreibung`

**Input:** `Ziegenrück.PRO`  
**Output:** `Ziegenrück_KPRO.csv`

---

In [ ]:
# export_kpro_to_csv.py (safer version)
from pathlib import Path
import re, csv

dataset_dir  = Path(r"C:\Users\raah\Desktop\Project_ZR\Ziegenrück")
dataset_name = "Ziegenrück"
pro_path     = dataset_dir / f"{dataset_name}.PRO"
csv_path     = dataset_dir / f"{dataset_name}_KPRO.csv"

lines = pro_path.read_text(encoding="cp1252", errors="ignore").splitlines()

# find [KPRO]
start = next((i for i,ln in enumerate(lines) if ln.strip().upper().startswith("[KPRO]")), None)
if start is None:
    raise RuntimeError("No [KPRO] section found.")

rows = []
i = start + 1
while i < len(lines):
    ln = lines[i]
    if re.match(r"^\s*\[[^\]]+\]\s*$", ln):  # next section
        break
    if re.match(r"^\s*N\s*=\s*\d+\s*$", ln, flags=re.IGNORECASE):
        i += 1; continue
    # Ignore any framelines or old pretty lines just in case
    if "|" in ln or re.match(r"^[\s\-\+\|\.\=]+$", ln):
        i += 1; continue
    toks = re.split(r"\s+", ln.strip())
    if toks and toks[0].isdigit():
        rows.append(toks)
    i += 1

# Write CSV: ; delimiter, quotes around fields when needed, CRLF (Windows), UTF-8
with open(csv_path, "w", newline="", encoding="utf-8") as f:
    w = csv.writer(f, delimiter=";", quotechar='"', quoting=csv.QUOTE_MINIMAL, lineterminator="\r\n")
    w.writerow(["ID","OBID","dtA","dtE","dt_min","SimStart","t_hhmm","D_mm","NID_min","HWID","tSkal","QSkal","Beschreibung"])
    for r in rows:
        w.writerow(r)

print(f"[OK] Exported {len(rows)} rows -> {csv_path}")